# Task 2 – Evaluating an object detector (YOLOv8 on KITTI)
1. Load the KITTI_Selection scenes and their ground truth (GT)
2. Run YOLOv8 and keep only cars
3. IoU between every detection and every GT car
4. Match detections ↔ GT (greedy vs. Hungarian – and why gating matters)
5. Precision / recall for every scene
6. Discussion: why do errors happen?

Theory and results: `docs/EXAM_NOTES.md`

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "main.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from kitti import load_dataset
from detector import YoloDetector, COCO_CAR
from matching import iou_matrix, match_greedy, match_hungarian, precision_recall
from evaluation import evaluate_scene
from visualize import draw_scene

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", 90)

def show(img, title="", w=14):
    plt.figure(figsize=(w, w * img.shape[0] / img.shape[1] + 0.3))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title(title); plt.axis("off"); plt.show()

scenes = load_dataset(ROOT / "data/KITTI_Selection", ROOT / "data/KITTI_full_labels")
by_id = {s.id: s for s in scenes}
print(len(scenes), "scenes,", sum(len(s.cars) for s in scenes), "GT cars")
pd.DataFrame([dict(scene=s.id, gt_cars=len(s.cars), image=cv2.imread(str(s.image_path)).shape[:2]) for s in scenes]).T

## 1. One scene with its ground truth
Change `SCENE` to look at another one. Green = GT car (with distance in metres, needed in Task 3).

In [ ]:
SCENE = "006211"

s = by_id[SCENE]
img = cv2.imread(str(s.image_path))
vis = img.copy()
for j, g in enumerate(s.cars):
    b = g.box.astype(int)
    cv2.rectangle(vis, tuple(b[:2]), tuple(b[2:]), (0, 200, 0), 2)
    cv2.putText(vis, f"{j}: {g.distance:.0f}m", (b[0], b[1] - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)
show(vis, f"{SCENE}: {len(s.cars)} GT cars")
print("camera matrix K:\n", s.K)

## 2. Run YOLOv8
YOLO is trained on COCO (80 classes). We keep only class 2 = **car**.

In [ ]:
detector = YoloDetector(str(ROOT / "models/yolov8m.pt"), conf=0.25)   # downloads the weights on first use
all_dets = detector(img)
print("all YOLO detections:", Counter(d.cls_name for d in all_dets))

vis = img.copy()
for d in all_dets:
    b = d.box.astype(int)
    col = (0, 0, 255) if d.cls_id == COCO_CAR else (200, 200, 200)
    cv2.rectangle(vis, tuple(b[:2]), tuple(b[2:]), col, 2)
    cv2.putText(vis, f"{d.cls_name} {d.conf:.2f}", (b[0], b[1] - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.4, col, 1)
show(vis, "red = car (kept), grey = other classes (dropped)")

## 3. IoU matrix
Rows = car detections, columns = GT cars. IoU = area(A∩B) / area(A∪B).

In [ ]:
cars = [d for d in all_dets if d.cls_id == COCO_CAR]
M = iou_matrix(np.array([d.box for d in cars]).reshape(-1, 4), np.array([g.box for g in s.cars]).reshape(-1, 4))

plt.figure(figsize=(8, 6))
plt.imshow(M, cmap="Blues", vmin=0, vmax=1)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if M[i, j] > 0.01:
            plt.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8,
                     color="white" if M[i, j] > 0.6 else "black")
plt.xlabel("GT car"); plt.ylabel("detection"); plt.colorbar(label="IoU"); plt.title(f"IoU matrix {SCENE}")
plt.show()

## 4. Matching: greedy vs. Hungarian
Each detection may match at most one GT car and vice versa, and the IoU must be at least 0.5.
* **greedy**: by confidence, take the best free GT box
* **Hungarian**: maximise the total IoU (optimal assignment)
* **Hungarian without gating**: may give up a good pair for two weak ones that are then
  thrown away (look at scene 006211!)

In [ ]:
THR = 0.5
conf = np.array([d.conf for d in cars])
variants = {"greedy": match_greedy(M, conf, THR),
            "hungarian (gated)": match_hungarian(M, THR),
            "hungarian without gating": match_hungarian(M, THR, gate=False)}
for name, pairs in variants.items():
    print(f"{name:26s} TP={len(pairs):2d}  pairs (det, gt): {sorted(pairs)}")
only = set(variants["hungarian (gated)"]) - set(variants["hungarian without gating"])
print("\nlost without gating:", [(i, j, round(M[i, j], 2)) for i, j in only])

## 5. Evaluated scene
Green = GT found, orange = GT missed (FN), blue = correct detection (TP, with IoU),
red = wrong detection (FP), grey dashed = KITTI *DontCare* region.

In [ ]:
r = evaluate_scene(s, all_dets, iou_thr=THR, method="hungarian")
show(draw_scene(img, r), "")
p, rc = r.precision, r.recall
print(f"TP={r.tp}  FP={r.fp}  FN={r.fn}   precision={p:.2f}   recall={rc:.2f}")
pd.DataFrame(r.det_rows + [x for x in r.gt_rows if x["status"] == "FN"])[["det", "gt", "status", "conf", "iou", "reason"]]

## 6. All scenes
YOLO runs on every image (about 10 s on a CPU).

In [ ]:
results, images = [], {}
for sc in scenes:
    im = cv2.imread(str(sc.image_path))
    images[sc.id] = im
    results.append(evaluate_scene(sc, detector(im), iou_thr=THR, method="hungarian"))

for res in results:
    show(draw_scene(images[res.scene.id], res), w=12)

## 7. Precision and recall per scene
*n/a* = undefined (0/0): scenes without GT cars have no recall; a scene with no detections has no precision.

In [ ]:
f = lambda x: None if x is None else round(x, 2)
table = pd.DataFrame([dict(scene=x.scene.id, GT=len(x.scene.cars), det=len(x.dets), TP=x.tp, FP=x.fp, FN=x.fn,
                           precision=f(x.precision), recall=f(x.recall), mean_IoU=f(x.mean_iou_tp)) for x in results])
TP, FP, FN = table.TP.sum(), table.FP.sum(), table.FN.sum()
P, R = precision_recall(TP, FP, FN)
print(f"ALL scenes (micro): TP={TP} FP={FP} FN={FN}  precision={P:.2f}  recall={R:.2f}")
print(f"mean over scenes (macro): precision={table.precision.mean():.2f}  recall={table.recall.mean():.2f}")
table.fillna("n/a")

### IoU of every detection

In [ ]:
dets_df = pd.DataFrame([row for x in results for row in x.det_rows])
dets_df[["scene", "det", "conf", "status", "gt", "iou", "reason"]]

## 8. Why do errors happen?
### Missed cars vs. occlusion
KITTI's GT box of an occluded car covers the **whole** car (also the hidden part), YOLO boxes only the
**visible** part, so the IoU drops below 0.5 although the car was found.

In [ ]:
gt_df = pd.DataFrame([row for x in results for row in x.gt_rows])
occ = gt_df.groupby("occlusion").apply(lambda g: pd.Series(dict(
    cars=len(g), recall_iou05=(g.status == "found").mean(), recall_iou03=(g.iou >= 0.3).mean())))
display(occ.round(2))
occ[["recall_iou05", "recall_iou03"]].plot.bar(figsize=(7, 3.5), color=["#1f3b63", "#9aa5b1"])
plt.xticks(range(len(occ)), ["0 visible", "1 partly", "2 largely", "3 unknown"][:len(occ)], rotation=0)
plt.ylabel("recall"); plt.ylim(0, 1.05); plt.title("Recall per KITTI occlusion level"); plt.show()

print("Missed cars (FN):")
gt_df[gt_df.status == "FN"][["scene", "gt", "iou", "occlusion", "truncation", "height_px", "distance_m", "reason"]]

### Wrong detections (FP)

In [ ]:
fp = dets_df[dets_df.status == "FP"]
fp.reason.str.split(" \\(").str[0].value_counts().plot.barh(figsize=(8, 3.5), color="#c0392b")
plt.xlabel("count"); plt.title("Reasons for false positives"); plt.gca().invert_yaxis(); plt.show()

### Worst scenes
006315: precision 0.22, but almost all "false" detections are real parked cars that are not in the GT.
006211 / 006291: crowded rows of parked cars → occlusion → low IoU.

In [ ]:
for sid in ("006315", "006211", "006291"):
    res = next(x for x in results if x.scene.id == sid)
    show(draw_scene(images[sid], res), w=12)

## 9. Precision–recall vs. confidence threshold
Higher threshold → fewer, more reliable detections (precision ↑, recall ↓).
`python ablation.py` also compares the models n/s/m, matching strategies and IoU thresholds
(`results/ablation.md`).

In [ ]:
det_low = YoloDetector(str(ROOT / "models/yolov8m.pt"), conf=0.01)
raw = {sc.id: det_low(images[sc.id]) for sc in scenes}
pts = []
for c in np.linspace(0.02, 0.95, 40):
    rs = [evaluate_scene(sc, [d for d in raw[sc.id] if d.conf >= c], THR, "hungarian") for sc in scenes]
    p, r_ = precision_recall(sum(x.tp for x in rs), sum(x.fp for x in rs), sum(x.fn for x in rs))
    if p is not None:
        pts.append((c, p, r_))
pts = np.array(pts)
plt.figure(figsize=(5, 4))
plt.plot(pts[:, 2], pts[:, 1], lw=2, color="#1f3b63")
k = np.argmin(abs(pts[:, 0] - 0.25))
plt.scatter(pts[k, 2], pts[k, 1], color="#c0392b", zorder=5, label="conf 0.25")
plt.xlabel("recall"); plt.ylabel("precision"); plt.xlim(0, 1); plt.ylim(0, 1.02); plt.grid(alpha=.3); plt.legend()
plt.title("YOLOv8m, IoU ≥ 0.5"); plt.show()